# Importancia de la topología informacional de BTC

Análisis causal y reproducible de la utilidad de `BTC_y` como coordenada del simplex transaccional y como descriptor de regímenes futuros de volatilidad.

El notebook reproduce las etapas A–K del análisis exploratorio: dependencia lineal y no lineal, lags, heterocedasticidad, volatilidad futura, regresión, clasificación de tres y cinco regímenes y contrastes anidados de utilidad informacional.

**Contrato:** sin `shuffle`, features disponibles hasta `t`, targets posteriores a `t`, umbrales ajustados sólo en train, frontera train/test purgada y walk-forward expansivo con `gap` temporal.

## 1. Dependencias

Colab ya incluye NumPy, pandas, SciPy, scikit-learn y Matplotlib. Se aseguran únicamente los lectores Parquet y LightGBM.

In [ ]:
%pip install -q lightgbm pyarrow

## 2. Drive y configuración

- Dejá `RUN_PIPELINE=True` para estimar todo.
- Cambialo a `False` para leer resultados ya guardados sin volver a entrenar.
- `RUN_WALK_FORWARD=False` permite una primera prueba rápida sólo con holdout.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Neural/NPP/Cripto')
PRICE_PATH = Path('/content/drive/MyDrive/0626p.parquet')
SIMPLEX_PATH = Path('/content/drive/MyDrive/0626dfyp.parquet')
OUTPUT_DIR = PROJECT_DIR / 'topologia_informacional_BTC'
DRIVE_SCRIPT = PROJECT_DIR / 'analisis_topologia_informacional_btc.py'

RUN_PIPELINE = True
RUN_WALK_FORWARD = True
BACKEND = 'lightgbm'  # 'lightgbm' o 'histgb'
SEED = 42
FOLDS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Resultados:', OUTPUT_DIR)

## 3. Motor científico versionado

Se utiliza primero la copia guardada junto al notebook en Drive. Si no existe, se descarga la versión exacta del commit científico publicado en GitHub.

In [ ]:
import urllib.request

PINNED_COMMIT = '3b640eaa35f9f84def677dea8f83afdb670aa985'
RAW_SCRIPT_URL = (
    'https://raw.githubusercontent.com/Miguithub/NPP/'
    f'{PINNED_COMMIT}/analisis_topologia_informacional_btc.py'
)
RUNTIME_SCRIPT = Path('/content/analisis_topologia_informacional_btc.py')

if DRIVE_SCRIPT.exists():
    RUNTIME_SCRIPT.write_bytes(DRIVE_SCRIPT.read_bytes())
    print('Motor cargado desde Drive:', DRIVE_SCRIPT)
else:
    urllib.request.urlretrieve(RAW_SCRIPT_URL, RUNTIME_SCRIPT)
    print('Motor descargado desde el commit:', PINNED_COMMIT)

assert RUNTIME_SCRIPT.stat().st_size > 10_000, 'El script descargado está incompleto.'

## 4. Ejecución completa o reanudación desde CSV

Esta celda estima el pipeline sólo cuando `RUN_PIPELINE=True`. Los resultados quedan persistidos inmediatamente en `OUTPUT_DIR`; en otra sesión podés poner el interruptor en `False` y continuar directamente con la lectura.

In [ ]:
import subprocess
import sys

if RUN_PIPELINE:
    command = [
        sys.executable, str(RUNTIME_SCRIPT),
        '--price-path', str(PRICE_PATH),
        '--simplex-path', str(SIMPLEX_PATH),
        '--output-dir', str(OUTPUT_DIR),
        '--backend', BACKEND,
        '--seed', str(SEED),
        '--folds', str(FOLDS),
    ]
    if not RUN_WALK_FORWARD:
        command.append('--skip-walk-forward')
    subprocess.run(command, check=True)
else:
    required = OUTPUT_DIR / '13_utilidad_topologia.csv'
    if not required.exists():
        raise FileNotFoundError(
            f'No hay resultados guardados en {OUTPUT_DIR}. Activá RUN_PIPELINE.'
        )
    print('Se reutilizan los resultados existentes; no se reentrena.')

## 5. Funciones de lectura

Cada CSV contiene columnas `experiment`, `model` y `feature_set` para que ninguna métrica quede separada del modelo que la produjo.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

def load_result(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

print('Archivos disponibles:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print('-', path.name)

## 6. Auditoría de datos y split

Verifica rango temporal, observaciones inválidas, tamaño de train/test, purga y ausencia de shuffle.

In [ ]:
audit = load_result('00_auditoria_datos.csv')
display(audit)

## 7. Etapas A–C — dependencia básica y tendencia compartida

Primero se comprueba si `BTC_y` se relaciona con el nivel de precio. La correlación parcial controla por el precio contemporáneo para detectar asociaciones generadas por persistencia o tendencia común.

In [ ]:
basic = load_result('01_dependencia_basica.csv')
derived = load_result('02_features_derivadas_vs_precio.csv')
display(basic)
display(derived.sort_values('mutual_information', ascending=False))

## 8. Etapas D–F — cambios, lags y naturaleza de la señal

Se distingue si `ΔBTC_y` informa dirección, magnitud, colas o varianza condicional. Una AUC cercana a 0.5 descarta dirección; una caída de MI al estandarizar por volatilidad apunta a información de régimen.

In [ ]:
relations = load_result('03_dBTC_y_vs_precio_retorno.csv')
lag_scan = load_result('04_barrido_lags_retorno.csv')
lag_decomposition = load_result('05_descomposicion_lag_4.csv')
display(relations)
display(lag_scan.sort_values('mutual_information', ascending=False).head(10))
display(lag_decomposition)

## 9. Etapa G — topología y volatilidad futura

El barrido lag × horizonte se calcula sólo sobre desarrollo. La estabilidad entre mitades permite detectar un pico aislado que no se repite temporalmente.

In [ ]:
future_vol = load_result('06_barrido_volatilidad_futura.csv')
halves = load_result('06b_estabilidad_mitades.csv')
display(
    future_vol.sort_values(['horizon', 'mutual_information'], ascending=[True, False])
    .groupby('horizon', as_index=False).head(3)
)
display(halves)
display(Image(filename=str(OUTPUT_DIR / 'fig_01_mi_lag_horizonte.png')))

## 10. Etapa H — regresión directa de volatilidad

Compara persistencia, regresión lineal y boosting con y sin `ΔBTC_y`. Si la mejora es prácticamente cero, la variable no debe venderse como regresor puntual aunque sí pueda servir como gate.

In [ ]:
reg_holdout = load_result('07_regresion_holdout.csv')
reg_walk = load_result('08_regresion_walk_forward.csv')
display(reg_holdout.sort_values('RMSE'))
if not reg_walk.empty:
    display(
        reg_walk.groupby(['model', 'feature_set'])[['MAE', 'RMSE', 'R2']]
        .agg(['mean', 'std'])
    )

## 11. Etapas I–J — clasificación de regímenes

Se comparan 3 regímenes, 5 regímenes con 4.5% en cada cola extrema y 5 quintiles libres. Macro-F1 y balanced accuracy son las métricas principales; accuracy sola puede ocultar el colapso de clases minoritarias.

In [ ]:
classification = load_result('09_clasificacion_holdout.csv')
display(
    classification.sort_values(['regime_scheme', 'macro_f1'], ascending=[True, False])
    [['regime_scheme', 'model', 'feature_set', 'accuracy',
      'balanced_accuracy', 'macro_f1', 'log_loss']]
)
display(Image(filename=str(OUTPUT_DIR / 'fig_02_macro_f1_holdout.png')))

## 12. Recall y F1 por régimen

Esta tabla revela si un promedio aparentemente bueno se obtuvo sin identificar los estados extremos.

In [ ]:
by_class = load_result('10_clasificacion_por_clase_holdout.csv')
display(
    by_class.sort_values(['regime_scheme', 'model', 'class_id'])
    [['regime_scheme', 'model', 'feature_set', 'class_name',
      'precision', 'recall', 'f1', 'support']]
)

## 13. Validación walk-forward

La media resume desempeño, pero el conteo de folds ganados muestra consistencia. Semillas y folds responden preguntas distintas: los folds prueban estabilidad temporal.

In [ ]:
walk = load_result('09b_clasificacion_walk_forward.csv')
if walk.empty:
    print('Walk-forward desactivado. Activá RUN_WALK_FORWARD y volvé a ejecutar el pipeline.')
else:
    walk_summary = (
        walk.groupby(['regime_scheme', 'model', 'feature_set'])
        [['balanced_accuracy', 'macro_f1', 'log_loss']]
        .agg(['mean', 'std'])
    )
    display(walk_summary)

## 14. Importancia de variables

La importancia se agrega por familia. No debe interpretarse como causalidad ni como porcentaje de explicación porque existen variables correlacionadas.

In [ ]:
importance = load_result('11_importancia_features_holdout.csv')
family_importance = (
    importance.groupby(['regime_scheme', 'model', 'feature_set', 'feature_family'], as_index=False)
    ['importance_share'].sum()
    .sort_values(['regime_scheme', 'model', 'importance_share'], ascending=[True, True, False])
)
display(family_importance)
display(
    importance.sort_values('importance_share', ascending=False)
    [['regime_scheme', 'model', 'feature', 'feature_family',
      'importance_method', 'importance_share']].head(40)
)

## 15. Etapa K — utilidad incremental de la topología informacional

Esta es la conclusión central. Cada modelo con `BTC_y` se compara con un control de la misma arquitectura. Los cuatro `gain_*` están orientados de modo que un valor positivo favorece la topología.

In [ ]:
utility = load_result('13_utilidad_topologia.csv')
utility_summary = load_result('14_resumen_utilidad_topologia.csv')
display(utility.sort_values(['regime_scheme', 'contrast', 'split']))
display(utility_summary)
display(Image(filename=str(OUTPUT_DIR / 'fig_03_utilidad_incremental.png')))

## 16. Lectura científica automática

El bloque resume dirección, magnitud y estabilidad del aporte sin convertir una diferencia predictiva en afirmación causal.

In [ ]:
incremental = utility[utility['contrast'].str.startswith('incremental', na=False)].copy()
holdout_incremental = incremental[incremental['split'] == 'holdout']
walk_incremental = incremental[incremental['split'].str.startswith('fold_')]

print('HOLDOUT — utilidad incremental por esquema')
display(
    holdout_incremental[['regime_scheme', 'contrast', 'model', 'baseline_model',
                         'gain_balanced_accuracy', 'gain_macro_f1',
                         'gain_log_loss', 'relative_gain_macro_f1']]
)

if not walk_incremental.empty:
    stability = (
        walk_incremental.groupby(['regime_scheme', 'contrast'], as_index=False)
        .agg(
            folds=('split', 'count'),
            mean_gain_macro_f1=('gain_macro_f1', 'mean'),
            wins_macro_f1=('gain_macro_f1', lambda x: int((x > 0).sum())),
            mean_gain_log_loss=('gain_log_loss', 'mean'),
        )
    )
    print('WALK-FORWARD — estabilidad temporal')
    display(stability)

print(
    'Interpretación: hay evidencia favorable cuando el gain es positivo en holdout '
    'y se repite en la mayoría de los folds. Esto valida utilidad predictiva '
    'incremental de esta operacionalización de primer orden; no demuestra causalidad.'
)

## 17. Manifiesto reproducible

Conserva configuración, backend, ventanas, columnas, tamaños muestrales y contrato temporal de la corrida.

In [ ]:
manifest_path = OUTPUT_DIR / 'manifest.json'
with manifest_path.open(encoding='utf-8') as handle:
    manifest = json.load(handle)
display(manifest)

## Conclusión de alcance

El experimento evalúa si una coordenada informacional del simplex aporta capacidad de identificación de regímenes. Sus variables son componentes NPP de primer orden: accesibles y transferibles, pero todavía sujetos a mejorar con microestructura, volatilidad condicional, liquidez efectiva y mediciones directas de resistencia sistémica.

El uso posterior propuesto es un gate probabilístico de cinco estados: `[P_XL, P_L, P_M, P_H, P_XH]`, no una predicción mecánica de precio.